In [0]:
from pyspark.sql import functions as F

# ==========================================================
# Configuration
# ==========================================================

SILVER_TABLE     = "finance_intelligence_hub.silver.stock_prices"
COMPANIES_TABLE  = "finance_intelligence_hub.silver.companies"
GOLD_VIEW        = "finance_intelligence_hub.gold.fact_stock_prices"

PRICE_DECIMALS = 4   # standard precision for equity prices in BI tools

print("="*80)
print("Stock Prices Gold View")
print("="*80)



spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_VIEW} AS
SELECT
    sp.ticker,
    sp.trade_date,
    CAST(ROUND(sp.open_price, {PRICE_DECIMALS}) AS DECIMAL(18,{PRICE_DECIMALS}))            AS open_price,
    CAST(ROUND(sp.high_price, {PRICE_DECIMALS}) AS DECIMAL(18,{PRICE_DECIMALS}))            AS high_price,
    CAST(ROUND(sp.low_price, {PRICE_DECIMALS}) AS DECIMAL(18,{PRICE_DECIMALS}))             AS low_price,
    CAST(ROUND(sp.close_price, {PRICE_DECIMALS}) AS DECIMAL(18,{PRICE_DECIMALS}))           AS close_price,
    CAST(ROUND(sp.adjusted_close_price, {PRICE_DECIMALS}) AS DECIMAL(18,{PRICE_DECIMALS}))  AS adjusted_close_price,
    sp.volume
FROM {SILVER_TABLE} sp
LEFT SEMI JOIN {COMPANIES_TABLE} c
    ON sp.ticker = c.ticker
""")

print(f"Gold view {GOLD_VIEW} created/refreshed successfully.")

# ==========================================================
# Quick sanity check
# ==========================================================

row_count = spark.sql(f"SELECT COUNT(*) FROM {GOLD_VIEW}").first()[0]
print(f"Gold view row count: {row_count:,}")

spark.sql(f"DESCRIBE {GOLD_VIEW}").show(truncate=False)